# SPSD v4 — Colab Test Harness
**Semantic Prompt Structural Distillation — SLM Edition**

Architecture: Tier 1 rule-based guards → Qwen2.5-1.5B-Instruct Q4_K_M (CPU) → Tier 1b fallback check

**Free-tier Colab compatible.** CPU runtime only, ~900MB RAM for model.

Run cells in order: Install → Download → Load → Test

In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────────────────────
# llama-cpp-python: pre-built CPU wheel (no compilation needed)
# huggingface-hub: for model download

!pip install -q huggingface-hub

# Use pre-built wheel for free-tier Colab CPU (avoids 10min compile)
import platform, sys
py = f"{sys.version_info.major}{sys.version_info.minor}"
print(f'Python {sys.version_info.major}.{sys.version_info.minor} | {platform.machine()}')

# Install pre-built llama-cpp-python wheel (CPU, no AVX512 required)
!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

print('\nInstall complete.')

In [ ]:
# ── CELL 2: Upload SPSD v4 source files ─────────────────────────────────────
# Upload spsd_v4.py and ale_prompt.py using the cell below,
# OR clone from your repo if hosted:
# !git clone https://github.com/YOUR_REPO/spsd.git && cd spsd

from google.colab import files
print('Upload spsd_v4.py and ale_prompt.py')
uploaded = files.upload()
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
# ── CELL 3: Download model ───────────────────────────────────────────────────
# Qwen2.5-1.5B-Instruct Q4_K_M
# Size: ~986MB  |  RAM usage: ~900MB  |  Inference: ~80-180ms on Colab CPU

import spsd_v4

model_path = spsd_v4.download_model(dest_dir='/content/models')
print(f'Model ready at: {model_path}')

In [ ]:
# ── CELL 4: Load model ──────────────────────────────────────────────────────
# Load once — reuse for all subsequent cells.
# Also reloads spsd_v4 and ale_prompt to ensure latest uploaded files are active.
# If you re-upload spsd_v4.py or ale_prompt.py, re-run this cell before testing.

import importlib
import spsd_v4
import ale_prompt
importlib.reload(spsd_v4)
importlib.reload(ale_prompt)

spsd_v4.load_model(model_path=model_path)

print('Model loaded.')
print(f'SHORT_PROMPT_WORD_LIMIT : {spsd_v4.SHORT_PROMPT_WORD_LIMIT}')
print(f'MIN_NET_TOKEN_SAVING    : {spsd_v4.MIN_NET_TOKEN_SAVING}')
print(f'ALE_HEADER_OVERHEAD     : {spsd_v4.ALE_HEADER_OVERHEAD_TOKENS}')
print(f'Compression ratio gate  : REMOVED — net saving gate only')

In [ ]:
# ── CELL 5: Tier 1 guard tests (no model needed) ────────────────────────────
# Tier 1 now only blocks: short prompts (<=15 words) and hard safety phrases.
# Medical / legal / financial / code are NO LONGER blocked — they are tagged
# and sent to the SLM with a domain marker.

import importlib
importlib.reload(spsd_v4)

tier1_cases = [
    # Should passthrough — too short
    ('Hi',                                True,  'short_prompt'),
    ('Where is my order',                 True,  'short_prompt'),
    ('How do I reset my password',        True,  'short_prompt'),

    # Should passthrough — safety critical
    ('I am having chest pain and cannot breathe please help me', True, 'safety_critical'),
    ('I think I may have taken an overdose of my medication',    True, 'safety_critical'),

    # Should NOT passthrough — domain tagged but distillable
    ('I have been taking ibuprofen 400mg twice daily for two weeks and also have hypertension is that safe', False, None),
    ('Can I sue my landlord for refusing to fix my heating for three months I have it in writing',           False, None),
    ('Should I put my savings into index funds or bonds given I am in my mid thirties',                     False, None),
    ('I keep getting a null pointer exception in my java checkout service when users apply discount codes',  False, None),

    # Should NOT passthrough — general support
    ('Hi I am so sorry to bother you but I ordered something yesterday and I cannot find my tracking number and my kids are crying', False, None),
]

print('TIER 1 GUARD TESTS — minimal gate (short + safety only)')
print('='*65)
passed = 0
for prompt, exp_pt, exp_reason in tier1_cases:
    pt, reason = spsd_v4.tier1_check(prompt)
    domain = spsd_v4._detect_domain(prompt)
    ok = pt == exp_pt
    if ok: passed += 1
    status = 'PASS' if ok else 'FAIL'
    dom_str = f' [{domain}]' if domain else ''
    print(f'[{status}] pt={str(pt):5s} reason={str(reason):16s}{dom_str}')
    print(f'       "{prompt[:70]}"')

print(f'\n{passed}/{len(tier1_cases)} passed')

In [ ]:
# ── CELL 6: Core SLM distillation tests ─────────────────────────────────────
# All prompts are realistic user messages — verbose, with filler and context.
# Every prompt is >15 words so tier1 passes them through to the SLM.
# Expected: most distill with token_saving > 0. Passthrough reasons are shown.

import importlib
importlib.reload(spsd_v4)

test_prompts = [
    # 1. Anxious support — tracking, active stressor, identifier
    "Hi I'm so sorry to bother you, I know you must be really busy, but I ordered something yesterday and I can't find the tracking number anywhere in my emails and my kids are crying in the background and I really need to know where this package is. The order number is TRK-00492. I'm incredibly stressed.",

    # 2. Medical domain — should tag [MEDICAL] and distill
    "I have been taking ibuprofen 400mg twice a day for about two weeks now for lower back pain and I am wondering whether that is too long and whether I should be worried about side effects, especially given that I also have mild hypertension that I manage with medication.",

    # 3. Legal domain — should tag [LEGAL] and distill
    "My landlord has been refusing to fix a serious damp and mould problem in my flat for over four months despite me reporting it in writing three times. I want to know whether I have grounds to take legal action and what my rights are as a tenant in this situation.",

    # 4. Code domain — should tag [CODE] and distill
    "I keep getting a NullPointerException in my Java checkout service every time a user tries to apply a discount code. The stack trace points to CartService.applyDiscount on line 42. I have already verified that the discount object itself is not null before it gets passed in, so I am confused about what is causing this.",

    # 5. Multi-question linked
    "I have two related questions. First, can you explain the difference between supervised and unsupervised machine learning? And then based on that explanation, which approach would be more appropriate for a project where I am trying to cluster customer purchase data but I do not have any predefined category labels to train on?",

    # 6. Polite subscription cancellation
    "Hello, I hope you are having a good day. I have been a customer for about three years now and have really enjoyed the service overall, but I think the time has come for me to cancel my subscription. Could you please help me understand what the cancellation process involves, whether there are any fees or penalties, and how long it will take to actually take effect?",

    # 7. Negative complaint with identifier
    "This is absolutely unacceptable. I placed order ORD-88821 over three weeks ago and I still have not received it. Every time I contact your support team I get a completely different answer about what has happened to it. I want a full refund and I want it processed immediately.",

    # 8. Financial — should tag [FINANCIAL] and distill
    "I am in my mid-thirties and I have about twenty thousand pounds sitting in a savings account earning very little interest. I have been thinking about moving it into some kind of investment but I am not sure whether index funds or bonds or something else would be more appropriate given my age and a medium risk tolerance.",
]

print('SLM DISTILLATION TESTS')
print('='*65)

distilled = 0
for i, prompt in enumerate(test_prompts, 1):
    print(f'\n[TEST {i}] {len(prompt.split())} words')
    result = spsd_v4.distill(prompt, model_path=model_path)
    print(result.summary())
    if not result.passthrough:
        distilled += 1
        reduction = (1 - len(result.compressed_prompt.split()) / len(prompt.split())) * 100
        print(f'Token reduction: ~{reduction:.0f}%')

print(f'\nDistilled: {distilled}/{len(test_prompts)}')

In [ ]:
# ── CELL 7: Latency benchmark ────────────────────────────────────────────────
# Realistic prompts — mix of tier1 exits (fast) and SLM distillations (slower).
# Target: SLM avg <200ms, p95 <350ms on free-tier Colab CPU.

import time

benchmark_prompts = [
    # SLM distillations — all >15 words, should compress
    "Hi I am really sorry to keep bothering you about this but I still cannot find the tracking number for my order TRK-00492 which I placed three days ago and have had no update on whatsoever.",
    "I have been taking ibuprofen for my back pain for about two weeks now and I have mild hypertension, is it safe to continue or should I switch to something else?",
    "My landlord has been refusing to fix the damp problem in my flat for four months, I have complained in writing three times, what are my tenant rights here?",
    "I keep getting a NullPointerException in my Java service on line 42 every time a discount code is applied, the discount object is confirmed non-null before the call.",
    "Hello I have been a loyal customer for three years and I would like to cancel my subscription and understand whether there are any early termination fees involved.",
    "I have about twenty thousand pounds in savings and I am in my mid thirties with a medium risk appetite, should I be putting this into index funds or bonds?",
    "This is completely unacceptable, I placed order ORD-88821 three weeks ago, I have contacted support four times and received a different story every single time, I want a refund.",
    "Can you explain the difference between supervised and unsupervised machine learning, and which one would suit a clustering task where I have no predefined labels?",

    # Tier 1 exits — short, fast
    "Hi",
    "Where is my order",
    "I am having chest pain and cannot breathe",
]

print('LATENCY BENCHMARK')
print('='*65)

latencies = []
for prompt in benchmark_prompts:
    t0 = time.time()
    r  = spsd_v4.distill(prompt, model_path=model_path)
    ms = (time.time() - t0) * 1000
    latencies.append((ms, r.tier, r.passthrough))
    outcome = 'PASSTHROUGH' if r.passthrough else f'DISTILLED saving={r.token_saving:+d}t'
    dom_str = f' [{r.domain}]' if r.domain else ''
    print(f'{ms:7.0f}ms  {outcome:28s}{dom_str}  "{prompt[:55]}"')

all_ms   = [x[0] for x in latencies]
slm_ms   = [x[0] for x in latencies if not x[2]]
tier1_ms = [x[0] for x in latencies if x[2]]

print(f'\nAll      avg={sum(all_ms)/len(all_ms):.0f}ms  p95={sorted(all_ms)[int(len(all_ms)*0.95)]:.0f}ms')
if slm_ms:
    print(f'SLM only avg={sum(slm_ms)/len(slm_ms):.0f}ms  p95={sorted(slm_ms)[int(len(slm_ms)*0.95)]:.0f}ms')
if tier1_ms:
    print(f'Tier 1   avg={sum(tier1_ms)/len(tier1_ms):.0f}ms  (rule-based, near-zero)')

In [ ]:
# ── CELL 8: ALE packet preview ───────────────────────────────────────────────
# Shows exactly what gets sent to the frontier LLM after distillation.
# Runs a fresh distill() — does NOT rely on any previous cell's result.
# This is the cell to use when checking header compression.

import importlib
import ale_prompt
importlib.reload(spsd_v4)
importlib.reload(ale_prompt)

preview_prompts = [
    # The key case — anxious tracking prompt with active stressor
    "Hi I'm so sorry to bother you, I know you must be really busy, but I ordered something yesterday and I can't find the tracking number anywhere in my emails and my kids are crying in the background and I really need to know where this package is. The order number is TRK-00492.",

    # Medical domain — should show D|MEDICAL| in header
    "I have been taking ibuprofen 400mg twice a day for about two weeks for lower back pain and I am wondering whether that is too long and whether I should be worried about side effects given I also have mild hypertension.",

    # Short prompt — should show P (passthrough)
    "Where is my order",
]

for prompt in preview_prompts:
    print(f'\nORIGINAL ({len(prompt.split())} words):')
    print(f'  {prompt[:100]}...' if len(prompt) > 100 else f'  {prompt}')
    result = spsd_v4.distill(prompt, model_path=model_path)
    ale_prompt.preview_packet(result)
    if not result.passthrough:
        orig_t = round(len(prompt.split()) / 0.75)
        print(f'Original ~{orig_t} tokens → packet shown above')

In [ ]:
# ── CELL 9: Full round-trip — distill → ALE → frontier LLM → display ────────
# Shows: original prompt / compressed prompt / metadata / frontier response
# / token counts / latency — everything in one view.
#
# Requires ANTHROPIC_API_KEY.
# Set it in Colab Secrets (key icon in left sidebar), then uncomment line below.

import os
import ale_prompt

# from google.colab import userdata
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

api_key = os.environ.get('ANTHROPIC_API_KEY')

if not api_key:
    print('ANTHROPIC_API_KEY not set — skipping frontier LLM test.')
    print('Set it in Colab Secrets (key icon in left sidebar) and uncomment above.')
else:
    try:
        import anthropic
    except ImportError:
        !pip install -q anthropic

    # ── Test prompts — swap in any prompt you want to evaluate ──
    test_prompts = [
        # Anxious support — tracking number, active stressor
        (
            "Hi I'm so sorry to bother you, I know you must be really busy, "
            "but I ordered something yesterday and I can't find the tracking number "
            "anywhere in my emails and my kids are crying in the background and I "
            "really need to know where this package is. The order number is TRK-00492."
        ),
        # Explanatory / idiom
        (
            "I've been trying to get into sourdough baking lately but I'm stuck on "
            "the autolyse step. Can you explain what it does and is it a dealbreaker "
            "if I skip it when I'm short on time?"
        ),
        # Passthrough — should forward original unchanged
        "What are the symptoms of type 2 diabetes?",
    ]

    for prompt in test_prompts:
        result = ale_prompt.round_trip(
            prompt=prompt,
            api_key=api_key,
            model='claude-sonnet-4-6',
            max_tokens=300,
            temperature=0.4,
            spsd_model_path=model_path,
        )
        result.display()

In [ ]:
# ── CELL 10: Batch evaluation ────────────────────────────────────────────────
# Realistic prompts — as a real user would write them.
# All support/domain prompts are >15 words so they reach the SLM.
# Expected: most distill. Short prompts correctly passthrough at tier1.

all_test_prompts = [
    # Support — anxious, identifier
    "Hi I am so sorry to keep bothering you about this but I still cannot find the tracking number for my order TRK-00492 which I placed three days ago and I have had absolutely no update on it whatsoever and my kids are getting upset.",
    "I placed an order three weeks ago with order number ORD-88821 and it still has not arrived. I have contacted your support team four times and received a completely different explanation every single time. I want a full refund and I want it processed today.",
    "I bought a pair of shoes from your site last week and unfortunately they do not fit at all. I would like to return them and get a refund but I cannot find any information about your return process on the website. Could you please explain how this works?",

    # Medical domain — tagged, distilled
    "I have been taking ibuprofen 400mg twice a day for about two weeks for persistent lower back pain and I am starting to wonder whether that is too long a course and whether there are any risks I should be aware of, particularly since I also have mild hypertension that I manage with blood pressure medication.",

    # Legal domain — tagged, distilled
    "My landlord has been refusing to address a serious damp and mould problem in my flat for over four months despite me raising it in writing on three separate occasions. I want to understand what my legal rights are as a tenant and whether I have grounds to take any kind of formal action against them.",

    # Financial domain — tagged, distilled
    "I am in my mid-thirties and I have approximately twenty thousand pounds sitting in a current account earning essentially no interest. I have been thinking about moving it into investments but I am genuinely not sure whether index funds or bonds or something else entirely would be more suitable given that I have a medium risk tolerance.",

    # Code domain — tagged, distilled
    "I keep getting a NullPointerException in my Java checkout service whenever a customer tries to apply a discount code during checkout. The stack trace shows the error is happening in CartService.applyDiscount on line 42. I have already checked and confirmed that the discount object being passed in is not null, so I cannot work out what is causing this.",

    # Explanatory
    "I have been trying to get into sourdough baking for the past month and I keep stumbling on the autolyse step in most recipes. I do not really understand what it is actually doing to the dough or why it matters, and I also want to know whether it is a dealbreaker if I skip it when I am short on time.",

    # Multi-question linked
    "I have two questions that are related to each other. First, can you explain the difference between supervised and unsupervised machine learning in plain terms? And then based on your answer to that, which approach would be better for a project where I am trying to group customer purchase data into segments but I do not have any predefined labels to work from?",

    # Casual conversational
    "Hey I have been going back and forth on this for a while now and I would genuinely love to hear a considered opinion. Do you think remote work is actually meaningfully better for productivity or is it mostly something people say because it sounds good, and does the evidence actually support it?",

    # Polite subscription
    "Hello, I hope you are doing well. I have been a customer for about three years now and have genuinely enjoyed using the service, but I have decided the time has come to cancel my subscription. Could you help me understand what the process is, whether there are any fees or penalties for cancelling before the end of a billing cycle, and how long it will take to take effect?",

    # Tier 1 — short, correctly passthrough
    "Hi",
    "Where is my order",
    "I am having chest pain and cannot breathe",
]

summary = spsd_v4.run_batch(all_test_prompts, model_path=model_path)
spsd_v4.print_batch_summary(summary)

In [ ]:
# ── CELL 11: Interactive REPL ────────────────────────────────────────────────
# Type any prompt and see the distillation result live.
# Type 'quit' to exit.
# Note: Colab may require clicking the input box that appears.

spsd_v4.run_interactive(model_path=model_path)